In [1]:
import sys
import numpy as np
from scipy.optimize import linprog
import itertools
import time
import copy
import random

In [2]:
random.seed(42)

In [38]:
tests = ['sc_157_0', 'sc_330_0', 'sc_1000_11', 'sc_5000_1', 'sc_10000_5', 'sc_10000_2']
thresholds = [(130000, 94402), (29, 24), (240, 147), (70, 31), (120, 64), (280, 167)]

In [4]:
def load_hyper_graph(test):
    with open(f"data/{test}") as file:
        lines = file.readlines()
        n, m = list(map(int, lines[0].split()))
        edges = list()
        cost = list()
        for i in range(m):
            arr = list(map(int, lines[1 + i].split()))
            cost.append(arr[0])
            edges.append(arr[1:])

        return n, m, cost, edges

Напишем checker для задачи, здесь это сделать легко

In [5]:
def check_set_cover(n, m, cost, edges, chosen):
    used = [0] * n
    for i in chosen:
        for j in edges[i]:
            used[j] += 1
    if min(used) == 0:
        raise Exception("Not all vertices are covered")

    result = 0
    for i in chosen:
        result += cost[i]
    return result

Проверим заглушку, выбирающую все множества, чтобы проверить корректность данных.

In [6]:
for test in tests:
    n, m, cost, edges = load_hyper_graph(test)
    chosen = range(m)
    result = check_set_cover(n, m, cost, edges, chosen)
    print(f"Result on {test}: {result}")

Result on sc_157_0: 1095050
Result on sc_330_0: 330
Result on sc_1000_11: 49830
Result on sc_5000_1: 253572
Result on sc_10000_5: 506804
Result on sc_10000_2: 506110


Попишем всякие жадники, например, просто на каждом этапе будем брать множество, которое покрывает набольшее число не покрытых вершин.

In [7]:
def passed_cnt(result, idx):
    if result <= thresholds[idx][1]:
        return 2
    elif result <= thresholds[idx][0]:
        return 1
    else:
        return 0

In [8]:
def tb_greedy(n, m, cost, edges):
    #tb = take biggest
    used = [0] * n
    chosen = list()
    while min(used) == 0:
        opt_inc = 0
        opt_idx = 0
        for i in range(m):
            inc = 0
            for j in edges[i]:
                if used[j] == 0:
                    inc += 1
                    
            if inc > opt_inc:
                opt_idx, opt_inc = i, inc
            elif inc == opt_inc and cost[i] < cost[opt_idx]:
                opt_idx = i 

        chosen.append(opt_idx)
        for j in edges[opt_idx]:
            used[j] += 1

    return chosen

In [9]:
def test_method(method, name):
    print(f"Checking {name}")
    score = 0
    for i, test in enumerate(tests):
        n, m, cost, edges = load_hyper_graph(test)
         
        start = time.time()
        chosen = method(n, m, cost, edges)
        end = time.time()

        elapsed = end - start
        print(f"Execution time: {elapsed:.4f} seconds")
        
        result = check_set_cover(n, m, cost, edges, chosen)
        passed = passed_cnt(result, i)
        
        if passed == 1:
            score += 3
        elif passed == 2:
            score += 5

        print(f"Target function {test}: {result}")
        print(f"Passed {test}: {passed}")

    print(f"Score: {score}")

In [10]:
test_method(tb_greedy, "tb_greedy")

Checking tb_greedy
Execution time: 0.0005 seconds
Target function sc_157_0: 99900
Passed sc_157_0: 1
Execution time: 0.0478 seconds
Target function sc_330_0: 30
Passed sc_330_0: 0
Execution time: 0.0081 seconds
Target function sc_1000_11: 813
Passed sc_1000_11: 0
Execution time: 0.0962 seconds
Target function sc_5000_1: 668
Passed sc_5000_1: 0
Execution time: 0.3814 seconds
Target function sc_10000_5: 1455
Passed sc_10000_5: 0
Execution time: 0.3165 seconds
Target function sc_10000_2: 3015
Passed sc_10000_2: 0
Score: 3


Прошел только первый порог!

Улучшим жадник следующим образом: будем нормировать стоимость ребра на количество добавлений.

In [11]:
def tbc_greedy(n, m, cost, edges):
    #tb = take biggest with cost
    used = [0] * n
    chosen = list()
    while min(used) == 0:
        opt_inc = max(cost)
        opt_idx = 0
        for i in range(m):
            inc = 0
            for j in edges[i]:
                if used[j] == 0:
                    inc += 1

            if inc == 0:
                continue
                
            inc = cost[i] / inc
            if inc < opt_inc:
                opt_idx, opt_inc = i, inc

        chosen.append(opt_idx)
        for j in edges[opt_idx]:
            used[j] += 1

    return chosen

In [12]:
test_method(tbc_greedy, "tbc_greedy")

Checking tbc_greedy
Execution time: 0.0005 seconds
Target function sc_157_0: 102100
Passed sc_157_0: 1
Execution time: 0.0336 seconds
Target function sc_330_0: 30
Passed sc_330_0: 0
Execution time: 0.0220 seconds
Target function sc_1000_11: 170
Passed sc_1000_11: 1
Execution time: 0.1729 seconds
Target function sc_5000_1: 36
Passed sc_5000_1: 1
Execution time: 0.6912 seconds
Target function sc_10000_5: 76
Passed sc_10000_5: 1
Execution time: 0.6247 seconds
Target function sc_10000_2: 192
Passed sc_10000_2: 1
Score: 15


Уже лучше!

А теперь попробуем воспользоваться линейной релаксацией подобно тому, что было на лекции. Однако будем на каждой итерации все вероятности умножать на фиксированный коэффициент, чтобы ускорить алгоритм.

In [20]:
def lin_relax(n, m, cost, edges):
    A = [[0] * m for i in range(n)]
    for i in range(m):
        for j in edges[i]:
            A[j][i] = 1
            
    b = [1] * n
    c = cost
    bounds = [(0, 1) for _ in range(m)]
    A_ub = [[-x for x in row] for row in A]
    b_ub = [-x for x in b]
    
    result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    return result.x

In [21]:
def augment_lr(n, m, cost, edges, upd=1.01):
    relax = lin_relax(n, m, cost, edges)
    lmbda = 1
    while True:
        used = [0] * n
        chosen = list()
        for i in range(m):
            valid = False
            for j in edges[i]:
                if used[j] == 0:
                    valid = True

            if not valid:
                continue
                
            prob = min(relax[i] * lmbda, 1)
            if random.random() <= prob:
                chosen.append(i)
                for j in edges[i]:
                    used[j] += 1

        if min(used) != 0:
            return chosen

        lmbda *= upd

In [22]:
test_method(augment_lr, "augment_lr")

Checking augment_lr
Execution time: 0.0019 seconds
Target function sc_157_0: 120200
Passed sc_157_0: 1
Execution time: 0.2548 seconds
Target function sc_330_0: 65
Passed sc_330_0: 0
Execution time: 0.0288 seconds
Target function sc_1000_11: 204
Passed sc_1000_11: 1
Execution time: 0.6218 seconds
Target function sc_5000_1: 46
Passed sc_5000_1: 1
Execution time: 1.5880 seconds
Target function sc_10000_5: 100
Passed sc_10000_5: 1
Execution time: 1.0557 seconds
Target function sc_10000_2: 235
Passed sc_10000_2: 1
Score: 15


Тоже самое(.

Идея: давайте каждый раз строить новую линейную релаксацию и брать максимальное ребро.
Эмперически я выяснил, что каждый раз можно брать некоторый батч ребер, давайте брать с каждым разом больше ребер.

In [23]:
def greedy_lr(n, m, cost, edges, cycle):
    used = [0] * n
    chosen = list()

    step = 1
    cur_iter = 1
    
    while min(used) == 0:
        step += 1
        A = [[0] * m for i in range(n)]
        for i in range(m):
            for j in edges[i]:
                if used[j] == 0:
                    A[j][i] = 1
                
        b = [1] * n
        for i in range(n):
            if used[i] > 0:
                b[i] = 0
        
        c = cost
        bounds = [(0, 1) for _ in range(m)]
        A_ub = [[-x for x in row] for row in A]
        b_ub = [-x for x in b]
        
        result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
        x = result.x

        if step % cycle == 0:
            cur_iter += 1 
            
        for rp in range(cur_iter):
            idx = np.argmax(x)
            x[idx] = 0
            valid = False
            for j in edges[idx]:
                if used[j] == 0:
                    valid = True
            if valid:
                chosen.append(idx)
                for j in edges[idx]:
                    used[j] += 1

    return chosen

Эмпирически получилось, что когда ребер мало - выгоднее чаще менять число итераций, а на большем числе - реже.

In [24]:
def greedy_lr_adapted(n, m, cost, edges):
    if m >= 5000:
        return greedy_lr(n, m, cost, edges, 4)
    else:
        return greedy_lr(n, m, cost, edges, 3)

In [25]:
test_method(greedy_lr_adapted, "greedy_lr_adapted")

Checking greedy_lr_adapted
Execution time: 0.0199 seconds
Target function sc_157_0: 94400
Passed sc_157_0: 2
Execution time: 0.5820 seconds
Target function sc_330_0: 28
Passed sc_330_0: 1
Execution time: 0.1531 seconds
Target function sc_1000_11: 146
Passed sc_1000_11: 2
Execution time: 1.6692 seconds
Target function sc_5000_1: 31
Passed sc_5000_1: 2
Execution time: 9.7408 seconds
Target function sc_10000_5: 69
Passed sc_10000_5: 1
Execution time: 13.6158 seconds
Target function sc_10000_2: 168
Passed sc_10000_2: 1
Score: 24


Попробуем подтюнить решение случайным удалением ребер, а далее жадным добавлением.

In [26]:
def tuned_greedy_lr_adapted(n, m, cost, edges):
    chosen = greedy_lr_adapted(n, m, cost, edges)
    used = [0] * n
    for i in chosen:
        for j in edges[i]:
            used[j] += 1

    init_used = used

    max_batch_size = 12
    for _ in range(2500):
        used = init_used.copy()
        
        del_siz = random.randint(1, min(len(chosen), max_batch_size))
        to_del = random.sample(chosen, del_siz)
        
        df = 0
        uncovered = 0
        
        for idx in to_del:
            df -= cost[idx]
            for j in edges[idx]:
                used[j] -= 1
                if used[j] == 0:
                    uncovered += 1

        to_ins = list()
        while uncovered > 0 and df < 0:
            opt_inc = float('inf')
            opt_idx = 0
            for i in range(m):
                inc = 0
                for j in edges[i]:
                    if used[j] == 0:
                        inc += 1
    
                if inc == 0:
                    continue
                    
                inc = cost[i] / inc
                if inc < opt_inc:
                    opt_idx, opt_inc = i, inc
    
            to_ins.append(opt_idx)
            df += cost[opt_idx]
            for j in edges[opt_idx]:
                used[j] += 1
                if used[j] == 1:
                    uncovered -= 1
        
        if df < 0:
            for x in to_del:
                chosen.remove(x)
            for x in to_ins:
                chosen.append(x)
                
            init_used = used

            
    return chosen

In [27]:
test_method(tuned_greedy_lr_adapted, "tuned_greedy_lr_adapted")

Checking tuned_greedy_lr_adapted
Execution time: 0.4549 seconds
Target function sc_157_0: 94400
Passed sc_157_0: 2
Execution time: 12.8019 seconds
Target function sc_330_0: 23
Passed sc_330_0: 2
Execution time: 4.9124 seconds
Target function sc_1000_11: 146
Passed sc_1000_11: 2
Execution time: 90.3468 seconds
Target function sc_5000_1: 31
Passed sc_5000_1: 2
Execution time: 171.3651 seconds
Target function sc_10000_5: 66
Passed sc_10000_5: 1
Execution time: 86.1646 seconds
Target function sc_10000_2: 168
Passed sc_10000_2: 1
Score: 26


Итоги текущих улучшений:
1. Жадник, который берет ребро с самой маленькой средней стоимостью вершины, сходу показал хорошие результаты.
2. Сэмплинг с "раздутой" вероятностью, полученной от линейной релаксации, не дал улучшений, возможно, структуры их решений получаются схожими.
3. Пришла интересная идея в голову, которая сильно улучшила результаты - это делать линейную релаксацию для итеративного выбора ребер. Данную идею можно чуть улучшить тем, что мы берем не одно ребро из релаксации, а сразу несколько. Идейно, кажется, что каждый перезапуск линейной релаксации сильно меняет структуру решения, поэтому логичнее брать с каждой итерацией больше из текущей релаксации, чтобы в конце быть более устойчивым решением. На маленьких графах, оказалось эффективнее быстрее увеличивать число взятых ребер, что довольно объяснимо, так как ребер мало и у нас нет времени на стохастику.
4. Стандартная модификация - это хорошее решение пытаться улучшать local-serch'ем. Я удалял какой - то батч ребер и пытался добавить новые, как в жаднике (чтобы делать больше итераций, я выбрал более быстрое решение).
5. Вышли близкие результаты к порогам на последних двух группах, можно попробовать еще какой - то способ подтюнить решение?

Вернемся к алгоритму, который решает линейную релаксацию после чего перестраивает задачу SetCover. Попробуем альтернативный подход, в котором мы выбираем не батч ребер, а добавляем шум в оценку каждого ребра, полученного через LP. Также берем лучшее ребро с учетом шума. Таким образом, мы сделаем больше стохастики у решения, потому что даже батчирование имеет сильно жадную структуру, которую LNS не может исправить.

Дополнительной оптимизацией будет ограничение количества ребер, чтобы итерации LP работали быстрее. Оставим какое - то количество самых дешевых ребер. Таким образом, еще и ответ LP может стать точнее.

P.S. 
Идеей замены батчирования на добавление шума я вдохновлялся из решения Глеба Костылева, правда я решил добавлять шум иначе. Постфактум это оказался мощный метод, который хорошо избегает локальные оптимумы.

In [28]:
def stochastic_lr(n, m, cost, edges, noise_val, bound_edge_cnt):
    selected = sorted(range(m), key=lambda i: cost[i])[:min(bound_edge_cnt, m)]

    cost = [cost[i] for i in selected]
    edges = [edges[i] for i in selected]
    m = len(selected)

    used = [0] * n
    chosen = list()

    while min(used) == 0:
        mapping = [-1] * m
        inv_mapping = []

        for i in range(m):
            for j in edges[i]:
                if used[j] == 0:
                    mapping[i] = len(inv_mapping)
                    inv_mapping.append(i)
                    break

        vertex_mapping = [-1] * n
        inv_vertex_mapping = []

        for i in range(n):
            if used[i] == 0:
                vertex_mapping[i] = len(inv_vertex_mapping)
                inv_vertex_mapping.append(i)

        new_n = len(inv_vertex_mapping)
        new_m = len(inv_mapping)

        A = [[0] * new_m for _ in range(new_n)]
        for i in inv_mapping:
            vi = mapping[i]
            for j in edges[i]:
                vj = vertex_mapping[j]
                if vj != -1:
                    A[vj][vi] = 1

        b = [1] * new_n
        c = [cost[i] for i in inv_mapping]

        bounds = [(0, 1) for _ in range(new_m)]
        A_ub = [[-x for x in row] for row in A]
        b_ub = [-x for x in b]

        noise = np.random.uniform(0, noise_val, size=new_m)

        result = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
        x = result.x + noise

        idx = np.argmax(x)
        real_idx = inv_mapping[idx]

        chosen.append(selected[real_idx])

        for j in edges[real_idx]:
            used[j] += 1

    return chosen

Сделаем несколько итераций стохастического жадного алгоритма и выберем лучшее решение.

In [40]:
def repeat_stochastic_lr(n, m, cost, edges, noise_val=0.15, bound_cnt=1000):
    best_chosen = list()
    best_cost = 0
    start = time.time()
    while True:
        cur = time.time()
        if cur - start > 280:
            break
            
        cur_chosen = stochastic_lr(n, m, copy.deepcopy(cost), copy.deepcopy(edges), noise_val, bound_cnt)
        cur_cost = check_set_cover(n, m, cost, edges, cur_chosen)

        if cur_cost < best_cost or len(best_chosen) == 0:
            best_chosen = cur_chosen
            best_cost = cur_cost

    return best_chosen

In [42]:
test_method(repeat_stochastic_lr, "stochastic_lr")

Checking stochastic_lr
Execution time: 280.0038 seconds
Target function sc_157_0: 94400
Passed sc_157_0: 2
Execution time: 280.3317 seconds
Target function sc_330_0: 24
Passed sc_330_0: 2
Execution time: 280.2070 seconds
Target function sc_1000_11: 146
Passed sc_1000_11: 2
Execution time: 280.1352 seconds
Target function sc_5000_1: 31
Passed sc_5000_1: 2
Execution time: 280.7247 seconds
Target function sc_10000_5: 64
Passed sc_10000_5: 2
Execution time: 281.5887 seconds
Target function sc_10000_2: 167
Passed sc_10000_2: 2
Score: 30


Прошли все тесты!